In [1]:
import sys
import os
import pandas as pd

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)


In [2]:
from src.geocoding import geocode_address
from src.pvgis_client import get_pvgis_hourly

In [3]:
data_dir = "../data"
files = [f for f in os.listdir(data_dir) if f.startswith("market_") and f.endswith(".csv")]
if not files: raise FileNotFoundError("Nerastas joks market_*.csv failas. Paleiskite 01_entsoe.ipynb.")
files = sorted(files, key=lambda f: os.path.getmtime(os.path.join(data_dir, f)))
latest_file = files[-1]

market_df = pd.read_csv(os.path.join(data_dir, latest_file), index_col=0, parse_dates=True)
print("Įkeltas ENSO‑E failas:", latest_file)
market_df.head()

Įkeltas ENSO‑E failas: market_2020-01-01_2020-01-07.csv


,price_eur_mwh,gen_solar_mw,gen_wind_onshore_mw,fc_solar_mw,fc_wind_onshore_mw,flow_lt_lv_mw,flow_lv_lt_mw,flow_lt_pl_mw,flow_pl_lt_mw,flow_lt_se_mw,flow_se_lt_mw
2020-01-01 00:00:00+02:00,NaN,NaN,NaN,NaN,NaN,0.0,196.5,482.16,0.0,0.0,43.75
2020-01-01 01:00:00+02:00,28.78,0.0,241.22,0.0,262.0,21.0,0.0,400.68,0.0,0.0,292.62
2020-01-01 02:00:00+02:00,28.45,0.0,261.60,0.0,236.0,0.0,44.0,400.73,0.0,0.0,215.43
2020-01-01 03:00:00+02:00,27.90,0.0,248.42,0.0,227.0,124.0,0.0,399.48,0.0,0.0,455.00
2020-01-01 04:00:00+02:00,27.52,0.0,219.89,0.0,219.0,185.5,0.0,260.73,0.0,0.0,364.55


In [4]:
lat, lon = geocode_address("Kaunas, Lithuania")
print("Koordinatės:", lat, lon)


Koordinatės: 54.8982139 23.9044817


In [5]:
df_pv = get_pvgis_hourly(lat, lon, 2020)
df_pv.head()

,pvgis_solar_kwh
time,
2020-01-01 00:11:00+00:00,0.0
2020-01-01 01:11:00+00:00,0.0
2020-01-01 02:11:00+00:00,0.0
2020-01-01 03:11:00+00:00,0.0
2020-01-01 04:11:00+00:00,0.0


In [6]:
df_pv = get_pvgis_hourly(lat, lon, 2020)

print("PVGIS duomenys gauti:")
df_pv.head()

PVGIS duomenys gauti:


,pvgis_solar_kwh
time,
2020-01-01 00:11:00+00:00,0.0
2020-01-01 01:11:00+00:00,0.0
2020-01-01 02:11:00+00:00,0.0
2020-01-01 03:11:00+00:00,0.0
2020-01-01 04:11:00+00:00,0.0


In [7]:
df_pv_local = df_pv.tz_convert("Europe/Vilnius")
df_pv_local.index = (
    df_pv_local.index
    .tz_localize(None)
    .round("h")
    .tz_localize("Europe/Vilnius",
                  ambiguous="NaT",
                  nonexistent="shift_forward")
)

In [8]:

df_pv_shifted = df_pv_local.copy()
df_pv_shifted.index = df_pv_shifted.index.map(lambda t: t.replace(year=2024))


In [9]:
df_pv_local = df_pv_local[~df_pv_local.index.duplicated(keep="first")]

df_pv_aligned = df_pv_local.reindex(market_df.index)

In [1]:
market_df["pvgis_solar_kwh"] = df_pv_aligned["pvgis_solar_kwh"]

market_df["latitude"] = lat
market_df["longitude"] = lon

NameError: name 'df_pv_aligned' is not defined

In [11]:
market_df.info()
market_df.head()

<class 'pandas.DataFrame'>
DatetimeIndex: 144 entries, 2020-01-01 00:00:00+02:00 to 2020-01-06 23:00:00+02:00
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   price_eur_mwh        143 non-null    float64
 1   gen_solar_mw         143 non-null    float64
 2   gen_wind_onshore_mw  143 non-null    float64
 3   fc_solar_mw          143 non-null    float64
 4   fc_wind_onshore_mw   143 non-null    float64
 5   flow_lt_lv_mw        144 non-null    float64
 6   flow_lv_lt_mw        144 non-null    float64
 7   flow_lt_pl_mw        144 non-null    float64
 8   flow_pl_lt_mw        144 non-null    float64
 9   flow_lt_se_mw        144 non-null    float64
 10  flow_se_lt_mw        144 non-null    float64
 11  pvgis_solar_kwh      142 non-null    float64
dtypes: float64(12)
memory usage: 14.6 KB


,price_eur_mwh,gen_solar_mw,gen_wind_onshore_mw,fc_solar_mw,fc_wind_onshore_mw,flow_lt_lv_mw,flow_lv_lt_mw,flow_lt_pl_mw,flow_pl_lt_mw,flow_lt_se_mw,flow_se_lt_mw,pvgis_solar_kwh
2020-01-01 00:00:00+02:00,NaN,NaN,NaN,NaN,NaN,0.0,196.5,482.16,0.0,0.0,43.75,NaN
2020-01-01 01:00:00+02:00,28.78,0.0,241.22,0.0,262.0,21.0,0.0,400.68,0.0,0.0,292.62,NaN
2020-01-01 02:00:00+02:00,28.45,0.0,261.60,0.0,236.0,0.0,44.0,400.73,0.0,0.0,215.43,0.0
2020-01-01 03:00:00+02:00,27.90,0.0,248.42,0.0,227.0,124.0,0.0,399.48,0.0,0.0,455.00,0.0
2020-01-01 04:00:00+02:00,27.52,0.0,219.89,0.0,219.0,185.5,0.0,260.73,0.0,0.0,364.55,0.0


In [12]:
output_path = os.path.join(data_dir, "market_with_pvgis.csv")
market_df.to_csv(output_path)

print("Galutinis dataset'as išsaugotas į:", output_path)

Galutinis dataset'as išsaugotas į: ../data\market_with_pvgis.csv
